In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
import pycountry_convert as pc
import plotly.express as px


In [2]:
# load data
project_root = Path.cwd().parent  # assumes you're in /notebooks
sys.path.append(str(project_root))
organization_df = pd.read_csv(os.path.join(project_root,'data/interim/organization_interim.csv'), on_bad_lines='warn', delimiter=';', keep_default_na=False)
country_names = pd.read_csv(os.path.join(project_root,'data/raw/cordisref-countries.csv'), on_bad_lines='warn', delimiter=';', keep_default_na=False)
country_names = country_names[country_names['language']=='en'].drop(columns='language')
country_names = country_names.rename(columns={'name':'countryName'})

In [3]:
# add country names
df = pd.merge(organization_df, country_names, left_on='country', right_on = 'euCode', how='inner')#.drop(columns='Unnamed: 0')
df['organizationCount'] = 1

# add continent names and codes (using pycountry-convert)
def convert_to_continent(isocode):
    try:
        return pc.country_alpha2_to_continent_code(isocode)
    except:
        #print(isocode, 'not a valid code, set manually')
        return
df['continent'] = df.isoCode.apply(convert_to_continent)
exceptions_dict = {'BQ': 'SA', 'VA': 'EU', 'XK': 'EU'}
df['continent'] = df['continent'].fillna(df['euCode'].map(exceptions_dict)) # manually add a few without ISO codes

continent_dict = dict(AF='Africa', AN='Antarctica', AS='Asia', EU='Europe', NA='North America', OC='Oceania', SA='South America')
df['continentName'] = df.continent.map(continent_dict)

In [4]:
# function that renames cities and orgs that are not in top 20 within their parent category to 'other'
def rename_bottom_contributors(_df, column, metric, N=10):
    # arguments: input dataframe, name of column that needs renaming ('city' or 'name'), name of metric to consider for ranking, number of names unchanged
    hierarchy = ['country', 'city', 'name']
    parent_column = hierarchy[hierarchy.index(column)-1]
    dfs = []
    for parent_name, group in _df.groupby(parent_column):
        top_index = group.groupby(column)[metric].sum().sort_values(ascending=False)[:N].index
        other_index = ~group[column].isin(top_index)
        group.loc[other_index, column] = 'Other '+{'city': 'cities', 'name':'organizations'}[column]
        dfs.append(group)
    return pd.concat(dfs)

In [5]:
# choose metric
metrics_list = ['ecContribution', 'netEcContribution', 'totalCost', 'organizationCount']
metric = metrics_list[1]

# filter for participation role
role = 'any' # any, coordinator, participant, thirdParty, associatedPartner

if role!='any':
    df = df[df.role==role]


df = rename_bottom_contributors(df, 'city', metric)

path = [px.Constant('All'), 'continentName', 'countryName', 'city']
if metric !='organizationCount':
    path.append('name')
    df = rename_bottom_contributors(df, 'name', metric)

In [6]:
df_plot = df
print('role:',role, '\nplotting', metric)
fig = px.treemap(df_plot, 
                 path=path, 
                 values=metric,
                 maxdepth=3
                 )
fig.update_traces(root_color="lightgrey")
fig.update_layout(margin = dict(t=50, l=25, r=25, b=25))
fig.show()

role: any 
plotting netEcContribution
